# PyTorch ile Derin Öğrenme · nn.Module: katmanlar, parametreler ve model yazmak

Macademia · Büyükler için

> **Colab'da çalıştırma:** Bu dosyayı [colab.research.google.com](https://colab.research.google.com) adresinde **Dosya → Not defteri yükle** ile aç. GPU için **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç. Hücreleri yukarıdan aşağı sırayla çalıştır (Shift + Enter). PyTorch Colab'da kuruludur; bilgisayarında çalıştırmak istersen `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # tekrarlanabilirlik
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Bir model sınıfı

In [ ]:
from torch import nn

class Siniflandirici(nn.Module):
    def __init__(self, n_in=2, gizli=16, n_out=1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, gizli), nn.ReLU(), nn.Linear(gizli, gizli), nn.ReLU(), nn.Linear(gizli, n_out))
    def forward(self, x):
        return self.net(x)

model = Siniflandirici()
print(model)
print(sum(p.numel() for p in model.parameters()), 'parametre')
for name, p in model.named_parameters():
    print(f'{name:14s} {tuple(p.shape)}')

## 2. Katman katman şekil izleme

In [ ]:
cnn = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(16 * 7 * 7, 10))
x = torch.randn(1, 1, 28, 28)
for layer in cnn:
    x = layer(x)
    n = sum(p.numel() for p in layer.parameters())
    print(f'{layer.__class__.__name__:10s} {str(tuple(x.shape)):18s} {n}')

## 3. Düz liste tuzağı

In [ ]:
class Yanlis(nn.Module):
    def __init__(self):
        super().__init__()
        self.katmanlar = [nn.Linear(4, 4) for _ in range(3)]          # düz liste: görünmez
class Dogru(nn.Module):
    def __init__(self):
        super().__init__()
        self.katmanlar = nn.ModuleList([nn.Linear(4, 4) for _ in range(3)])
print(len(list(Yanlis().parameters())), len(list(Dogru().parameters())))   # 0 6

## 4. Kaydet ve yükle

In [ ]:
torch.save(model.state_dict(), 'model.pt')
yeni = Siniflandirici()
yeni.load_state_dict(torch.load('model.pt'))
x = torch.randn(3, 2)
print(torch.allclose(model(x), yeni(x)))

## Alıştırmalar
1. `nn.Conv2d(3, 16, 3)`'ün parametre sayısını önce elle hesapla.
2. Yukarıdaki CNN'de ikinci `MaxPool2d`'yi kaldır: `Linear`'ın girişi ne olmalı?
3. `nn.LazyLinear` ile giriş boyunu otomatik bulduran bir model yaz.